In [89]:
import os
import pandas as pd
import itertools
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier

In [90]:
# Load the processed tournament data
tournament_file_path = os.path.join("..", "ProcessedData", "processed_tournament_data.csv")
tournament_data = pd.read_csv(tournament_file_path)

# Define target
y = tournament_data["Win"]


# Define features
features = ['Divison', 'T1Seed', 'T2Seed', 'SeedDifference', 'T1AverageScore', 
            'T1AverageFGM', 'T1AverageFGA', 'T1AverageFGM3', 'T1AverageFGA3', 'T1AverageFTM', 'T1AverageFTA', 'T1AverageOR', 'T1AverageDR', 
            'T1AverageAst', 'T1AverageTO', 'T1AverageStl', 'T1AverageBlk', 'T1AveragePF', 'T1AverageOpponentScore', 'T1AverageOpponentFGM', 
            'T1AverageOpponentFGA', 'T1AverageOpponentFGM3', 'T1AverageOpponentFGA3', 'T1AverageOpponentFTM', 'T1AverageOpponentFTA', 
            'T1AverageOpponentOR', 'T1AverageOpponentDR', 'T1AverageOpponentAst', 'T1AverageOpponentTO', 'T1AverageOpponentStl', 
            'T1AverageOpponentBlk', 'T1AverageOpponentPF', 'T1AveragePointDifference', 'T2AverageScore', 'T2AverageFGM', 'T2AverageFGA', 
            'T2AverageFGM3', 'T2AverageFGA3', 'T2AverageFTM', 'T2AverageFTA', 'T2AverageOR', 'T2AverageDR', 'T2AverageAst', 'T2AverageTO', 
            'T2AverageStl', 'T2AverageBlk', 'T2AveragePF', 'T2AverageOpponentScore', 'T2AverageOpponentFGM', 'T2AverageOpponentFGA', 
            'T2AverageOpponentFGM3', 'T2AverageOpponentFGA3', 'T2AverageOpponentFTM', 'T2AverageOpponentFTA', 'T2AverageOpponentOR', 
            'T2AverageOpponentDR', 'T2AverageOpponentAst', 'T2AverageOpponentTO', 'T2AverageOpponentStl', 'T2AverageOpponentBlk', 
            'T2AverageOpponentPF', 'T2AveragePointDifference', 'T1Rating', 'T2Rating', 'RatingDifference']

# Create dataframe to hold features
X = tournament_data[features]

In [91]:
# Split data into training data and validation data

# Use data prior to 2024 for training
train_X = X[tournament_data.Season < 2024]
train_y = y[tournament_data.Season < 2024]

# Use 2024 data for validation
val_X = X[tournament_data.Season == 2024]
val_y = y[tournament_data.Season == 2024]

# Create and train the Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=1)
rf_model.fit(train_X, train_y)

# Predict probabilities for the validation set
rf_val_predictions = rf_model.predict_proba(val_X)[:, 1]

# Evaluate predictions using Brier Score
val_brier_score = brier_score_loss(val_y, rf_val_predictions)
print(f"Random Forest Validation Brier Score: {val_brier_score:.4f}")


Random Forest Validation Brier Score: 0.1789


In [92]:
# Load stage 2 sample submission file to get the possible matchups for 2025
sample_submission_file_path = os.path.join("..", "RawData", "SampleSubmissionStage2.csv")
sample_submission = pd.read_csv(sample_submission_file_path)

# Extract season and team IDs from the sample submission file
sample_submission[['Season', 'T1TeamID', 'T2TeamID']] = sample_submission['ID'].str.split('_', expand=True).astype(int)

# Determine division based on team IDs (IDs >= 2000 are women, IDs < 2000 are men)
sample_submission['Division'] = (sample_submission['T1TeamID'] >= 2000).astype(int)